## A Notebook for Chinese Sentiment Classification using BERT

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-sentence-classification.png" />

In this notebook, we will use a pre-trained deep learning model to process some Chinese text. We will then use the output of that model to classify the text. The text is a list of sentences from Chinese herbal medicine reviews. And we will classify each sentence as either speaking "positively" about its subject or "negatively".

## Models: Sentence Sentiment Classification
Our goal is to create a model that takes a sentence (just like the ones in our dataset) and produces either 1 (indicating the sentence carries a positive sentiment) or a 0 (indicating the sentence carries a negative sentiment). We can think of it as looking like this:

<img src="https://jalammar.github.io/images/distilBERT/sentiment-classifier-1.png" />

Under the hood, the model is actually made up of two models.

* `bert-base-chinese` processes the sentence and passes along some information it extracted from it on to the next model. BERT is a powerful language representation model.
* The next model, a basic Logistic Regression model from scikit learn, will take in the result of BERT’s processing, and classify the sentence as either positive or negative (1 or 0, respectively).

The data we pass between the two models is a vector of size 768. We can think of this of vector as an embedding for the sentence that we can use for classification.


<img src="https://jalammar.github.io/images/distilBERT/distilbert-bert-sentiment-classifier.png" />

## Dataset
The dataset we will use in this example is `OpenModels/Chinese-Herbal-Medicine-Sentiment`, which contains sentences from Chinese herbal medicine reviews, each labeled as either positive (has the value 1) or negative (has the value 0).

## Installing the transformers library
Let's start by installing the huggingface transformers library so we can load our deep learning NLP model.

In [1]:
!pip install transformers

In [2]:
import numpy as np
import pandas as pd
import torch
import warnings

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, BertModel

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Importing the dataset
We'll use pandas to read the dataset and load it into a dataframe.

In [3]:
ds = load_dataset("OpenModels/Chinese-Herbal-Medicine-Sentiment")
df = ds["train"].to_pandas()

df = df.rename(columns={
    "review_text": "text",
    "sentiment_label": "label"
})

df = df[["text", "label"]].copy()
df["label"] = df["label"].map({
    "positive": 1,
    "negative": 0
})

df = df.dropna(subset=["text", "label"])
df = df[df["text"].astype(str).str.strip() != ""]
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

if len(df) > 3000:
    df = df.sample(n=3000, random_state=42).reset_index(drop=True)

print(df.head())
print(df["label"].value_counts())
print(df.shape)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/24.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.76M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211391 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23488 [00:00<?, ? examples/s]

                                                text  label
0                               当天到，快递很速度的噢，用起来也感觉很好    1.0
1             是杭州中医院同款的乌梅汤，酸酸甜甜很好喝，包装很正规，和医院买的一样，赞赞赞    1.0
2                           枸杞很新鲜，很漂亮，颗粒饱满，很干净，还会回购的    1.0
3  家人们谁懂啊，这个人参蜜片礼盒也太太太好看了吧，礼盒精致又高大上，简直是母亲节送礼必备！！！...    1.0
4  京东自营这个品牌的其他物品买过多次，这个品牌的百合干是第一次购买，质量与购物订单介绍的情况一...    1.0
label
1.0    2608
0.0     392
Name: count, dtype: int64
(3000, 2)


In [4]:
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")
model = BertModel.from_pretrained("bert-base-chinese")
model.to(device)
model.eval()

print("Model loaded.")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


In [5]:
def get_cls_embeddings(texts, tokenizer, model, device, batch_size=16, max_length=128):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        all_embeddings.append(cls_embeddings)

        if i % 200 == 0:
            print(f"Processed {i}/{len(texts)}")

    return np.vstack(all_embeddings)

In [6]:
texts = df["text"].tolist()
labels = df["label"].values

embeddings = get_cls_embeddings(
    texts=texts,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=16,
    max_length=128
)

print("Embeddings shape:", embeddings.shape)

Processed 0/3000
Processed 400/3000
Processed 800/3000
Processed 1200/3000
Processed 1600/3000
Processed 2000/3000
Processed 2400/3000
Processed 2800/3000
Embeddings shape: (3000, 768)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9516666666666667
              precision    recall  f1-score   support

         0.0       0.83      0.79      0.81        78
         1.0       0.97      0.98      0.97       522

    accuracy                           0.95       600
   macro avg       0.90      0.88      0.89       600
weighted avg       0.95      0.95      0.95       600

